# Zaid Hybrid IDS — Step 11H: ONNX Post-Optimization End-to-End Test v1

This notebook performs a post-optimization confirmatory benchmark of the complete online path:

`flow preparation → frozen random-forest trees in ONNX Runtime → TreeSHAP → grounded fast renderer → strict validator`

Scientific safeguards:

- Step 11G selected ONNX Runtime on all 120 Validation events before this notebook opens Explainer Test.
- The trained trees, 60-feature order, threshold `0.5`, TreeSHAP method, renderer, and validator remain unchanged.
- ONNX probability agreement, detector latency below 10 ms P95, and complete-path latency below 50 ms P95 are explicit acceptance conditions.
- Explainer Test was previously accessed in Steps 11D and 11F; this is a post-optimization confirmatory benchmark, not a new blind test.
- Timing starts before single-row preparation and ends after strict validation. Cold-start costs are reported separately.
- No training, threshold tuning, tree change, SHAP change, renderer change, or validator change occurs here.


In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)


Mounted at /content/drive


In [2]:
!pip install -q shap pyarrow onnxruntime


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 58.8 MB/s eta 0:00:00


In [3]:
from pathlib import Path
from time import perf_counter_ns
from collections import Counter
from datetime import datetime, timezone
import hashlib
import json
import platform
import re
import sys

import joblib
import numpy as np
import pandas as pd
import shap
import sklearn
import onnxruntime as ort

SEED = 20260831
THRESHOLD = 0.5
TIMING_ROUNDS = 10
PROBABILITY_TOLERANCE = 1e-4
np.random.seed(SEED)

PROJECT_ROOT = Path('/content/drive/MyDrive/Zaid_Hybrid_IDS')
SPLIT_DIR = PROJECT_ROOT / 'data/splits'
EXPLAINER_DIR = PROJECT_ROOT / 'data/explainer/qwen25_15b_strict_repair_v1'
RESULT_ROOT = PROJECT_ROOT / 'results'
MODEL_ROOT = PROJECT_ROOT / 'models'

RAW_TEST_PATH = SPLIT_DIR / 'test.parquet'
VALIDATION_TARGET_PATH = EXPLAINER_DIR / 'teacher_targets_validation.jsonl'
EXPLAINER_TEST_PATH = EXPLAINER_DIR / 'teacher_targets_test.jsonl'
FEATURE_MANIFEST_PATH = RESULT_ROOT / 'feature_sets_manifest.json'
SPLIT_MANIFEST_PATH = RESULT_ROOT / 'grouped_split_manifest.json'
FINAL_DETECTOR_MANIFEST_PATH = RESULT_ROOT / 'final_frozen_test_manifest.json'
FINAL_DETECTOR_METRICS_PATH = RESULT_ROOT / 'final_frozen_test_metrics.csv'
FAST_PATH_VALIDATION_MANIFEST_PATH = RESULT_ROOT / 'grounded_fast_path_v2/grounded_fast_path_manifest.json'
ONNX_VALIDATION_MANIFEST_PATH = RESULT_ROOT / 'grounded_fast_path_onnx_runtime_optimization_v1/onnx_runtime_validation_manifest.json'
ORIGINAL_TEST_MANIFEST_PATH = RESULT_ROOT / 'grounded_fast_path_end_to_end_test_v1_1/end_to_end_test_manifest.json'
STEP_11F_MANIFEST_PATH = RESULT_ROOT / 'grounded_fast_path_end_to_end_optimized_test_v1/optimized_end_to_end_test_manifest.json'
FROZEN_MODEL_PATH = MODEL_ROOT / 'random_forest_no_dst_port.joblib'
ONNX_MODEL_PATH = MODEL_ROOT / 'random_forest_no_dst_port_single_event_v1.onnx'
NF4_FALLBACK_DIR = MODEL_ROOT / 'student_qwen25_05b_distilled_quantized_v1/nf4_bnb'
FINAL_RESULT_DIR = RESULT_ROOT / 'grounded_fast_path_end_to_end_onnx_test_v1'
FINAL_MANIFEST_PATH = FINAL_RESULT_DIR / 'onnx_end_to_end_test_manifest.json'

for required_path in [
    RAW_TEST_PATH, VALIDATION_TARGET_PATH, EXPLAINER_TEST_PATH,
    FEATURE_MANIFEST_PATH, SPLIT_MANIFEST_PATH,
    FINAL_DETECTOR_MANIFEST_PATH, FINAL_DETECTOR_METRICS_PATH,
    FAST_PATH_VALIDATION_MANIFEST_PATH, ONNX_VALIDATION_MANIFEST_PATH,
    ORIGINAL_TEST_MANIFEST_PATH, STEP_11F_MANIFEST_PATH,
    FROZEN_MODEL_PATH, ONNX_MODEL_PATH, NF4_FALLBACK_DIR,
]:
    assert required_path.exists(), f'Missing required frozen artifact: {required_path}'

assert not FINAL_MANIFEST_PATH.exists(), (
    f'ONNX confirmatory Test benchmark was already completed: {FINAL_MANIFEST_PATH}. '
    'Do not rerun or overwrite it.'
)

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)

def read_jsonl(path):
    with path.open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

feature_manifest = read_json(FEATURE_MANIFEST_PATH)
split_manifest = read_json(SPLIT_MANIFEST_PATH)
detector_manifest = read_json(FINAL_DETECTOR_MANIFEST_PATH)
fast_validation_manifest = read_json(FAST_PATH_VALIDATION_MANIFEST_PATH)
onnx_validation_manifest = read_json(ONNX_VALIDATION_MANIFEST_PATH)
original_test_manifest = read_json(ORIGINAL_TEST_MANIFEST_PATH)
step_11f_manifest = read_json(STEP_11F_MANIFEST_PATH)
detector_metrics = pd.read_csv(FINAL_DETECTOR_METRICS_PATH).iloc[0].to_dict()

FEATURES = feature_manifest['no_dst_port_feature_set']
assert len(FEATURES) == 60
assert 'Dst Port' not in FEATURES
assert detector_manifest['threshold'] == THRESHOLD
assert detector_manifest['model_sha256'] == sha256_file(FROZEN_MODEL_PATH)
assert detector_manifest['test_file_sha256'] == split_manifest['outputs']['test']['sha256']
assert fast_validation_manifest['targets']['all_online_validation_targets_pass'] is True
assert onnx_validation_manifest['targets']['all_onnx_validation_targets_pass'] is True
assert onnx_validation_manifest['targets']['detector_single_event_p95_under_10ms'] is True
assert onnx_validation_manifest['targets']['integrated_single_event_p95_under_50ms'] is True
assert onnx_validation_manifest['source_model_sha256'] == detector_manifest['model_sha256']
assert onnx_validation_manifest['onnx_model_sha256'] == sha256_file(ONNX_MODEL_PATH)
assert original_test_manifest['model_sha256'] == detector_manifest['model_sha256']
assert step_11f_manifest['model_sha256'] == detector_manifest['model_sha256']

PIPELINE_SPEC = {
    'version': 'onnx_post_optimization_v1',
    'detector': 'same_frozen_random_forest_trees_exported_to_onnx',
    'source_detector_sha256': detector_manifest['model_sha256'],
    'onnx_model_sha256': sha256_file(ONNX_MODEL_PATH),
    'feature_set': 'no_dst_port_feature_set',
    'feature_count': 60,
    'threshold': THRESHOLD,
    'runtime_setting': 'ONNX Runtime CPU, sequential, one intra-op and one inter-op thread',
    'runtime_setting_selected_on': 'validation_only_step_11G',
    'probability_tolerance': PROBABILITY_TOLERANCE,
    'shap_method': 'TreeExplainer/tree_path_dependent/class_1/check_additivity_false',
    'top_features': 2,
    'renderer': 'grounded_fast_path_v2',
    'validator': 'strict_grounded_v2',
    'fallback': 'nf4_bnb_only_if_strict_validation_fails',
    'timing_rounds': TIMING_ROUNDS,
    'detector_latency_target_ms': 10.0,
    'pipeline_latency_target_ms': 50.0,
}
PIPELINE_SPEC_SHA256 = hashlib.sha256(
    json.dumps(PIPELINE_SPEC, sort_keys=True, separators=(',', ':')).encode('utf-8')
).hexdigest()

print('Frozen source detector:', FROZEN_MODEL_PATH)
print('Frozen ONNX detector:', ONNX_MODEL_PATH)
print('Source detector SHA-256:', detector_manifest['model_sha256'])
print('ONNX detector SHA-256:', sha256_file(ONNX_MODEL_PATH))
print('Frozen feature count:', len(FEATURES))
print('Frozen threshold:', THRESHOLD)
print('Pipeline specification SHA-256:', PIPELINE_SPEC_SHA256)
print('Validation ONNX detector P95 ms:', onnx_validation_manifest['detector_only_latency']['p95_ms'])
print('Validation ONNX End-to-End P95 ms:', onnx_validation_manifest['pipeline_latency']['end_to_end_ms']['p95_ms'])
print('Training or tuning in this notebook: False')
print('TEST WAS PREVIOUSLY ACCESSED; THIS IS A CONFIRMATORY DEPLOYMENT BENCHMARK.')
print('ONNX POST-OPTIMIZATION END-TO-END CONFIGURATION: FROZEN')


Frozen source detector: /content/drive/MyDrive/Zaid_Hybrid_IDS/models/random_forest_no_dst_port.joblib
Frozen ONNX detector: /content/drive/MyDrive/Zaid_Hybrid_IDS/models/random_forest_no_dst_port_single_event_v1.onnx
Source detector SHA-256: b0d0dd5ee8dafe1a0451958f1ac52247eb872217157e88631b83ae22fa59fe65
ONNX detector SHA-256: 598088e639fed58b154bbb86cf8eabd5749ffd6253dd0981d791077edc832a83
Frozen feature count: 60
Frozen threshold: 0.5
Pipeline specification SHA-256: f6c053c0467255c1d97c568f56bf338984d6da4689ed2d514aab004d5bc76b53
Validation ONNX detector P95 ms: 0.08703489999999994
Validation ONNX End-to-End P95 ms: 13.342029349999992
Training or tuning in this notebook: False
TEST WAS PREVIOUSLY ACCESSED; THIS IS A CONFIRMATORY DEPLOYMENT BENCHMARK.
ONNX POST-OPTIMIZATION END-TO-END CONFIGURATION: FROZEN


In [4]:
IP_PATTERN = re.compile(r'(?<![\d.])(?:\d{1,3}\.){3}\d{1,3}(?![\d.])')
PROHIBITED_PATTERN = re.compile(
    r'(?i)\b(exploit|payload|reverse shell|credential stuffing tool|malware family|attacker identity)\b'
)
ATTACK_SUBTYPE_PATTERN = re.compile(
    r'(?i)\b(ftp|ssh|brute[ -]?force|botnet|ddos|dos|infiltration|sql injection|xss|heartbleed)\b'
)
KNOWN_FEATURE_NAMES = sorted(FEATURES, key=len, reverse=True)

def normalize_output(text):
    text = text.strip().replace('\r\n', '\n').replace('\r', '\n')
    return '\n'.join(line.strip() for line in text.split('\n') if line.strip())

def supports_detector_decision(feature, decision):
    supports_attack = feature['direction'] == 'supports_attack'
    decision_is_attack = decision.lower() == 'attack'
    return supports_attack == decision_is_attack

def evidence_sentence(first, second, decision):
    first_supports = supports_detector_decision(first, decision)
    second_supports = supports_detector_decision(second, decision)
    if first_supports and second_supports:
        return (
            f"{first['name']} and {second['name']} provided the strongest local "
            "SHAP support for the detector decision."
        )
    if first_supports and not second_supports:
        return (
            f"{first['name']} supported the detector decision, while "
            f"{second['name']} opposed it according to local SHAP evidence."
        )
    if not first_supports and second_supports:
        return (
            f"{second['name']} supported the detector decision, while "
            f"{first['name']} opposed it according to local SHAP evidence."
        )
    return (
        f"{first['name']} and {second['name']} locally opposed the detector "
        "decision according to SHAP evidence."
    )

def render_grounded_fast(record):
    first, second = record['top_features'][:2]
    decision = str(record['predicted_class'])
    confidence = float(record['confidence'])
    return (
        f"Summary: {decision} with {confidence:.3f} confidence\n"
        f"Evidence: {evidence_sentence(first, second, decision)}\n"
        "Action: Review network, authentication, and host logs, corroborate before containment."
    )

def strict_validate(record, text):
    normalized = normalize_output(text)
    lines = normalized.splitlines()
    schema = (
        len(lines) == 3
        and lines[0].startswith('Summary:')
        and lines[1].startswith('Evidence:')
        and lines[2].startswith('Action:')
    )
    first, second = record['top_features'][:2]
    summary = lines[0] if len(lines) > 0 else ''
    evidence = lines[1] if len(lines) > 1 else ''
    action = lines[2] if len(lines) > 2 else ''
    decision = str(record['predicted_class'])
    exact_confidence = f"{float(record['confidence']):.3f}"
    decision_aligned = decision.lower() in summary.lower()
    confidence_aligned = exact_confidence in summary
    required_features = first['name'] in evidence and second['name'] in evidence
    allowed_names = {first['name'], second['name']}
    unsupported = any(name in evidence and name not in allowed_names for name in KNOWN_FEATURE_NAMES)
    decision_support = {
        supports_detector_decision(first, decision),
        supports_detector_decision(second, decision),
    }
    evidence_lower = evidence.lower()
    if decision_support == {True}:
        direction_aligned = 'support' in evidence_lower
    elif decision_support == {False}:
        direction_aligned = 'oppos' in evidence_lower
    else:
        direction_aligned = 'support' in evidence_lower and 'oppos' in evidence_lower
    action_lower = action.lower()
    action_grounded = (
        'review' in action_lower
        and 'log' in action_lower
        and 'corroborat' in action_lower
        and 'containment' in action_lower
    )
    prohibited = bool(
        IP_PATTERN.search(normalized)
        or PROHIBITED_PATTERN.search(normalized)
        or ATTACK_SUBTYPE_PATTERN.search(normalized)
    )
    valid = all([
        schema, decision_aligned, confidence_aligned, required_features,
        direction_aligned, action_grounded, not unsupported, not prohibited,
    ])
    return {
        'normalized': normalized,
        'schema_compliant': schema,
        'decision_aligned': decision_aligned,
        'confidence_aligned': confidence_aligned,
        'required_features_present': required_features,
        'direction_aligned': direction_aligned,
        'action_grounded': action_grounded,
        'unsupported_feature': unsupported,
        'prohibited_content': prohibited,
        'valid': valid,
    }

def extract_attack_shap(raw_values):
    if isinstance(raw_values, list):
        array = np.asarray(raw_values[1])
    else:
        raw_array = np.asarray(raw_values)
        if raw_array.ndim == 3 and raw_array.shape[-1] == 2:
            array = raw_array[:, :, 1]
        elif raw_array.ndim == 3 and raw_array.shape[0] == 2:
            array = raw_array[1]
        elif raw_array.ndim == 2:
            array = raw_array
        else:
            raise ValueError(f'Unexpected SHAP shape: {raw_array.shape}')
    assert array.shape[1] == len(FEATURES), array.shape
    return array

print('FROZEN RENDERER, VALIDATOR, AND SHAP ADAPTER: READY')


FROZEN RENDERER, VALIDATOR, AND SHAP ADAPTER: READY


In [5]:
data_load_start_ns = perf_counter_ns()
raw_test_df = pd.read_parquet(
    RAW_TEST_PATH,
    columns=[*FEATURES, 'Label', 'flow_group_id'],
)
X_source_df = raw_test_df[FEATURES].astype(np.float32)
X_source_np = np.ascontiguousarray(X_source_df.to_numpy(copy=False), dtype=np.float32)
data_load_ms = (perf_counter_ns() - data_load_start_ns) / 1_000_000.0

assert raw_test_df.shape == (156717, 62), raw_test_df.shape
assert np.isfinite(X_source_np).all()

initialization_start_ns = perf_counter_ns()
frozen_model = joblib.load(FROZEN_MODEL_PATH)
stored_n_jobs = frozen_model.n_jobs
frozen_model.n_jobs = 1
tree_explainer = shap.TreeExplainer(
    frozen_model,
    feature_perturbation='tree_path_dependent',
)
session_options = ort.SessionOptions()
session_options.intra_op_num_threads = 1
session_options.inter_op_num_threads = 1
session_options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
ort_session = ort.InferenceSession(
    str(ONNX_MODEL_PATH),
    sess_options=session_options,
    providers=['CPUExecutionProvider'],
)
onnx_input_name = ort_session.get_inputs()[0].name
onnx_output_names = [item.name for item in ort_session.get_outputs()]
assert len(onnx_output_names) >= 2, onnx_output_names
initialization_ms = (perf_counter_ns() - initialization_start_ns) / 1_000_000.0

feature_array = np.asarray(FEATURES)

def onnx_attack_probability(row_np):
    outputs = ort_session.run(None, {onnx_input_name: row_np})
    probabilities = np.asarray(outputs[1])
    assert probabilities.shape == (1, 2), probabilities.shape
    return float(probabilities[0, 1])

def run_integrated_event(expected_record):
    total_start_ns = perf_counter_ns()
    source_row = int(expected_record['source_test_row'])
    row_np = X_source_np[source_row:source_row + 1]
    row_df = X_source_df.iloc[[source_row]]
    after_prepare_ns = perf_counter_ns()

    attack_probability = onnx_attack_probability(row_np)
    predicted_class = 'Attack' if attack_probability >= THRESHOLD else 'Benign'
    detector_confidence = attack_probability if predicted_class == 'Attack' else 1.0 - attack_probability
    after_detector_ns = perf_counter_ns()

    raw_shap = tree_explainer.shap_values(row_df, check_additivity=False)
    attack_shap = extract_attack_shap(raw_shap)[0]
    ordered_indices = np.argsort(np.abs(attack_shap))[::-1][:5]
    top_features = [
        {
            'name': str(feature_array[index]),
            'value': float(row_np[0, index]),
            'shap_value': float(attack_shap[index]),
            'direction': 'supports_attack' if attack_shap[index] > 0 else 'opposes_attack',
        }
        for index in ordered_indices
    ]
    after_shap_ns = perf_counter_ns()

    runtime_record = {
        'event_id': expected_record['event_id'],
        'source_test_row': source_row,
        'predicted_class': predicted_class,
        'confidence': detector_confidence,
        'top_features': top_features,
    }
    explanation = render_grounded_fast(runtime_record)
    after_render_ns = perf_counter_ns()
    check = strict_validate(runtime_record, explanation)
    after_validate_ns = perf_counter_ns()

    expected_names = [item['name'] for item in expected_record['top_features'][:2]]
    runtime_names = [item['name'] for item in top_features[:2]]
    expected_decision = str(expected_record['predicted_class'])
    expected_confidence = float(expected_record['confidence'])

    return {
        'event_id': expected_record['event_id'],
        'source_test_row': source_row,
        'research_original_label': expected_record.get('research_original_label'),
        'flow_group_id': int(expected_record['flow_group_id']),
        'predicted_class': predicted_class,
        'attack_probability': attack_probability,
        'detector_confidence': detector_confidence,
        'expected_confidence': expected_confidence,
        'decision_fidelity': predicted_class == expected_decision,
        'confidence_delta': abs(detector_confidence - expected_confidence),
        'confidence_fidelity': abs(detector_confidence - expected_confidence) <= PROBABILITY_TOLERANCE,
        'top2_feature_fidelity': runtime_names == expected_names,
        'runtime_top2_features': runtime_names,
        'expected_top2_features': expected_names,
        'explanation': check['normalized'],
        'strict_valid': check['valid'],
        'fallback_required': not check['valid'],
        'prepare_ms': (after_prepare_ns - total_start_ns) / 1_000_000.0,
        'detector_ms': (after_detector_ns - after_prepare_ns) / 1_000_000.0,
        'shap_and_selection_ms': (after_shap_ns - after_detector_ns) / 1_000_000.0,
        'renderer_ms': (after_render_ns - after_shap_ns) / 1_000_000.0,
        'validator_ms': (after_validate_ns - after_render_ns) / 1_000_000.0,
        'end_to_end_ms': (after_validate_ns - total_start_ns) / 1_000_000.0,
    }

validation_records = read_jsonl(VALIDATION_TARGET_PATH)
assert len(validation_records) == 120
assert all(record['split'] == 'validation' for record in validation_records)

validation_rows = np.asarray([int(record['source_test_row']) for record in validation_records])
validation_np = X_source_np[validation_rows]
sklearn_probabilities = frozen_model.predict_proba(validation_np)[:, 1]
onnx_probabilities = np.asarray(
    ort_session.run(None, {onnx_input_name: validation_np})[1]
)[:, 1]
validation_probability_delta = float(np.max(np.abs(sklearn_probabilities - onnx_probabilities)))
validation_decision_agreement = bool(np.array_equal(
    sklearn_probabilities >= THRESHOLD,
    onnx_probabilities >= THRESHOLD,
))
assert validation_probability_delta <= PROBABILITY_TOLERANCE
assert validation_decision_agreement

zero_row = np.zeros((1, len(FEATURES)), dtype=np.float32)
for _ in range(20):
    _ = onnx_attack_probability(zero_row)
    _ = tree_explainer.shap_values(pd.DataFrame(zero_row, columns=FEATURES), check_additivity=False)

validation_integrated = [run_integrated_event(record) for record in validation_records]
assert sum(item['strict_valid'] for item in validation_integrated) == 120
assert sum(item['decision_fidelity'] for item in validation_integrated) == 120
assert sum(item['confidence_fidelity'] for item in validation_integrated) == 120
assert sum(item['top2_feature_fidelity'] for item in validation_integrated) == 120

print('Raw frozen detector Test rows:', len(raw_test_df))
print('Data load ms (offline):', data_load_ms)
print('Model, TreeExplainer, and ONNX session initialization ms (offline):', initialization_ms)
print('Validation maximum sklearn-to-ONNX probability delta:', validation_probability_delta)
print('Validation decision agreement:', validation_decision_agreement)
print('Validation integrated strict-valid: 120/120')
print('Validation decision/confidence/top-2 fidelity: 120/120')
print('Test access remains deferred until this Validation gate passes.')
print('ONNX INTEGRATED VALIDATION GATE: PASSED')


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


Raw frozen detector Test rows: 156717
Data load ms (offline): 1885.783998
Model, TreeExplainer, and ONNX session initialization ms (offline): 124.096673
Validation maximum sklearn-to-ONNX probability delta: 7.836539375194462e-07
Validation decision agreement: True
Validation integrated strict-valid: 120/120
Validation decision/confidence/top-2 fidelity: 120/120
Test access remains deferred until this Validation gate passes.
ONNX INTEGRATED VALIDATION GATE: PASSED


In [6]:
print('ONNX POST-OPTIMIZATION TEST BENCHMARK: BEGIN')
test_target_sha256 = sha256_file(EXPLAINER_TEST_PATH)
test_records = read_jsonl(EXPLAINER_TEST_PATH)

assert len(test_records) == 122, len(test_records)
assert all(record['split'] == 'test' for record in test_records)
assert len({record['event_id'] for record in test_records}) == 122
assert all(0 <= int(record['source_test_row']) < len(X_source_df) for record in test_records)
print('Unique Test flow groups:', len({record['flow_group_id'] for record in test_records}))

test_rows = np.asarray([int(record['source_test_row']) for record in test_records])
test_np = X_source_np[test_rows]
test_sklearn_probabilities = frozen_model.predict_proba(test_np)[:, 1]
test_onnx_probabilities = np.asarray(
    ort_session.run(None, {onnx_input_name: test_np})[1]
)[:, 1]
test_probability_delta = float(np.max(np.abs(test_sklearn_probabilities - test_onnx_probabilities)))
test_decision_agreement = bool(np.array_equal(
    test_sklearn_probabilities >= THRESHOLD,
    test_onnx_probabilities >= THRESHOLD,
))

all_timing_records = []
first_round_results = []
for timing_round in range(TIMING_ROUNDS):
    round_results = []
    for expected_record in test_records:
        result = run_integrated_event(expected_record)
        result['timing_round'] = timing_round + 1
        round_results.append(result)
        all_timing_records.append({
            key: result[key]
            for key in [
                'event_id', 'timing_round', 'prepare_ms', 'detector_ms',
                'shap_and_selection_ms', 'renderer_ms', 'validator_ms',
                'end_to_end_ms',
            ]
        })
    if timing_round == 0:
        first_round_results = round_results
    else:
        assert all(
            current['explanation'] == baseline['explanation']
            for current, baseline in zip(round_results, first_round_results)
        )
    print(f'Completed timing round {timing_round + 1}/{TIMING_ROUNDS}')

event_df = pd.DataFrame(first_round_results)
timing_df = pd.DataFrame(all_timing_records)

strict_valid_rate = float(event_df['strict_valid'].mean())
fallback_rate = float(event_df['fallback_required'].mean())
decision_fidelity_rate = float(event_df['decision_fidelity'].mean())
confidence_fidelity_rate = float(event_df['confidence_fidelity'].mean())
top2_feature_fidelity_rate = float(event_df['top2_feature_fidelity'].mean())

component_columns = [
    'prepare_ms', 'detector_ms', 'shap_and_selection_ms',
    'renderer_ms', 'validator_ms', 'end_to_end_ms',
]
summary_rows = []
for component in component_columns:
    values = timing_df[component].to_numpy(dtype=np.float64)
    summary_rows.append({
        'component': component,
        'calls': len(values),
        'mean_ms': float(values.mean()),
        'p50_ms': float(np.quantile(values, 0.50)),
        'p95_ms': float(np.quantile(values, 0.95)),
        'p99_ms': float(np.quantile(values, 0.99)),
        'max_ms': float(values.max()),
    })
latency_summary_df = pd.DataFrame(summary_rows)
detector_summary = latency_summary_df.loc[latency_summary_df['component'].eq('detector_ms')].iloc[0]
end_to_end_summary = latency_summary_df.loc[latency_summary_df['component'].eq('end_to_end_ms')].iloc[0]

detector_accuracy = float(detector_metrics['accuracy'])
quality_target_pass = strict_valid_rate >= 0.95
detector_accuracy_target_pass = detector_accuracy >= 0.95
detector_latency_target_pass = float(detector_summary['p95_ms']) < 10.0
pipeline_latency_target_pass = float(end_to_end_summary['p95_ms']) < 50.0
probability_equivalence_pass = test_probability_delta <= PROBABILITY_TOLERANCE and test_decision_agreement
all_final_targets_pass = bool(
    quality_target_pass
    and detector_accuracy_target_pass
    and detector_latency_target_pass
    and pipeline_latency_target_pass
    and probability_equivalence_pass
    and fallback_rate == 0.0
    and decision_fidelity_rate == 1.0
    and confidence_fidelity_rate == 1.0
    and top2_feature_fidelity_rate == 1.0
)

print('Explainer Test events:', len(test_records))
print('Explainer Test labels for QA only:', dict(Counter(record['research_original_label'] for record in test_records)))
print('Maximum sklearn-to-ONNX Test probability delta:', test_probability_delta)
print('Sklearn-to-ONNX Test decision agreement:', test_decision_agreement)
print('Strict-valid rate:', strict_valid_rate)
print('Fallback rate:', fallback_rate)
print('Decision/confidence/top-2 fidelity rates:', decision_fidelity_rate, confidence_fidelity_rate, top2_feature_fidelity_rate)
print('Frozen detector accuracy:', detector_accuracy)
print('\nIntegrated latency summary:')
print(latency_summary_df.to_string(index=False))
print('\nDetector P95 ms:', float(detector_summary['p95_ms']))
print('Detector target <10 ms:', detector_latency_target_pass)
print('End-to-End P95 ms:', float(end_to_end_summary['p95_ms']))
print('End-to-End target <50 ms:', pipeline_latency_target_pass)
print('ALL ONNX TEST TARGETS PASS:', all_final_targets_pass)
assert all_final_targets_pass
print('ONNX POST-OPTIMIZATION END-TO-END TEST: COMPLETE')


ONNX POST-OPTIMIZATION TEST BENCHMARK: BEGIN
Unique Test flow groups: 116


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


Completed timing round 1/10
Completed timing round 2/10
Completed timing round 3/10
Completed timing round 4/10
Completed timing round 5/10
Completed timing round 6/10
Completed timing round 7/10
Completed timing round 8/10
Completed timing round 9/10
Completed timing round 10/10
Explainer Test events: 122
Explainer Test labels for QA only: {'FTP-BruteForce': 60, 'SSH-Bruteforce': 60, 'Benign': 2}
Maximum sklearn-to-ONNX Test probability delta: 7.943792975639852e-07
Sklearn-to-ONNX Test decision agreement: True
Strict-valid rate: 1.0
Fallback rate: 0.0
Decision/confidence/top-2 fidelity rates: 1.0 1.0 1.0
Frozen detector accuracy: 0.9999744762852786

Integrated latency summary:
            component  calls  mean_ms   p50_ms    p95_ms    p99_ms    max_ms
           prepare_ms   1220 0.463639 0.380997  0.744680  3.263075  6.400336
          detector_ms   1220 0.234860 0.184132  0.262199  2.255072  6.238643
shap_and_selection_ms   1220 6.984626 6.027219 15.315440 22.811476 37.791435
     

In [7]:
FINAL_RESULT_DIR.mkdir(parents=True, exist_ok=True)
EVENT_RESULTS_PATH = FINAL_RESULT_DIR / 'onnx_end_to_end_test_event_results.parquet'
TIMING_CALLS_PATH = FINAL_RESULT_DIR / 'onnx_end_to_end_test_timing_calls.parquet'
LATENCY_SUMMARY_PATH = FINAL_RESULT_DIR / 'onnx_end_to_end_test_latency_summary.csv'
OUTPUTS_PATH = FINAL_RESULT_DIR / 'onnx_end_to_end_test_outputs.jsonl'

event_df.to_parquet(EVENT_RESULTS_PATH, index=False, compression='zstd')
timing_df.to_parquet(TIMING_CALLS_PATH, index=False, compression='zstd')
latency_summary_df.to_csv(LATENCY_SUMMARY_PATH, index=False)

with OUTPUTS_PATH.open('w', encoding='utf-8') as handle:
    for result in first_round_results:
        handle.write(json.dumps({
            'event_id': result['event_id'],
            'source_test_row': result['source_test_row'],
            'predicted_class': result['predicted_class'],
            'detector_confidence': result['detector_confidence'],
            'runtime_top2_features': result['runtime_top2_features'],
            'explanation': result['explanation'],
            'strict_valid': result['strict_valid'],
            'fallback_required': result['fallback_required'],
        }, ensure_ascii=False) + '\n')

manifest = {
    'schema_version': '1.0',
    'stage': 'onnx_post_optimization_confirmatory_end_to_end_test',
    'interpretation': 'Confirmatory deployment benchmark after Validation-only ONNX selection in Step 11G; not a new blind Test.',
    'onnx_validation_manifest': str(ONNX_VALIDATION_MANIFEST_PATH),
    'original_test_manifest': str(ORIGINAL_TEST_MANIFEST_PATH),
    'step_11f_manifest': str(STEP_11F_MANIFEST_PATH),
    'evaluated_at_utc': datetime.now(timezone.utc).isoformat(),
    'pipeline_specification': PIPELINE_SPEC,
    'pipeline_specification_sha256': PIPELINE_SPEC_SHA256,
    'scope': 'held_out_explainer_test_122',
    'test_target_file': str(EXPLAINER_TEST_PATH),
    'test_target_sha256': test_target_sha256,
    'raw_detector_test_file': str(RAW_TEST_PATH),
    'raw_detector_test_sha256': detector_manifest['test_file_sha256'],
    'source_model_file': str(FROZEN_MODEL_PATH),
    'source_model_sha256': detector_manifest['model_sha256'],
    'onnx_model_file': str(ONNX_MODEL_PATH),
    'onnx_model_sha256': sha256_file(ONNX_MODEL_PATH),
    'threshold': THRESHOLD,
    'test_events': len(test_records),
    'timing_rounds': TIMING_ROUNDS,
    'timed_single_event_calls': len(timing_df),
    'offline_data_load_ms': data_load_ms,
    'offline_model_explainer_and_onnx_initialization_ms': initialization_ms,
    'test_numerical_equivalence': {
        'max_probability_delta': test_probability_delta,
        'decision_agreement': test_decision_agreement,
        'tolerance': PROBABILITY_TOLERANCE,
    },
    'quality': {
        'strict_valid_rate': strict_valid_rate,
        'fallback_rate': fallback_rate,
        'decision_fidelity_rate': decision_fidelity_rate,
        'confidence_fidelity_rate': confidence_fidelity_rate,
        'top2_feature_fidelity_rate': top2_feature_fidelity_rate,
    },
    'latency': {
        row['component']: {
            key: float(row[key]) if key != 'calls' else int(row[key])
            for key in ['calls', 'mean_ms', 'p50_ms', 'p95_ms', 'p99_ms', 'max_ms']
        }
        for row in summary_rows
    },
    'targets': {
        'frozen_detector_accuracy': detector_accuracy,
        'detector_accuracy_at_least_0_95': detector_accuracy_target_pass,
        'explanation_valid_rate_at_least_0_95': quality_target_pass,
        'detector_single_event_p95_under_10ms': detector_latency_target_pass,
        'integrated_single_event_p95_under_50ms': pipeline_latency_target_pass,
        'probability_and_decision_equivalence': probability_equivalence_pass,
        'zero_test_fallbacks': fallback_rate == 0.0,
        'all_onnx_test_targets_pass': all_final_targets_pass,
    },
    'nf4_fallback': {
        'available': NF4_FALLBACK_DIR.exists(),
        'path': str(NF4_FALLBACK_DIR),
        'invoked_on_test_events': int(event_df['fallback_required'].sum()),
        'latency_not_hidden': True,
    },
    'training_performed': False,
    'threshold_or_hyperparameter_tuning_on_test': False,
    'test_previously_accessed': True,
    'post_test_policy': 'Report beside Steps 11D and 11F as a post-optimization confirmatory benchmark.',
    'software': {
        'python': sys.version,
        'platform': platform.platform(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'sklearn': sklearn.__version__,
        'shap': shap.__version__,
        'onnxruntime': ort.__version__,
    },
    'outputs': {
        'event_results': str(EVENT_RESULTS_PATH),
        'timing_calls': str(TIMING_CALLS_PATH),
        'latency_summary': str(LATENCY_SUMMARY_PATH),
        'explanations': str(OUTPUTS_PATH),
    },
}

with FINAL_MANIFEST_PATH.open('x', encoding='utf-8') as handle:
    json.dump(manifest, handle, ensure_ascii=False, indent=2)

print('Saved:', EVENT_RESULTS_PATH)
print('Saved:', TIMING_CALLS_PATH)
print('Saved:', LATENCY_SUMMARY_PATH)
print('Saved:', OUTPUTS_PATH)
print('Saved:', FINAL_MANIFEST_PATH)
print('ALL ONNX TEST TARGETS PASS:', all_final_targets_pass)
print('ONNX POST-OPTIMIZATION END-TO-END TEST: SAVED AND FROZEN')


Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/grounded_fast_path_end_to_end_onnx_test_v1/onnx_end_to_end_test_event_results.parquet
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/grounded_fast_path_end_to_end_onnx_test_v1/onnx_end_to_end_test_timing_calls.parquet
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/grounded_fast_path_end_to_end_onnx_test_v1/onnx_end_to_end_test_latency_summary.csv
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/grounded_fast_path_end_to_end_onnx_test_v1/onnx_end_to_end_test_outputs.jsonl
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/grounded_fast_path_end_to_end_onnx_test_v1/onnx_end_to_end_test_manifest.json
ALL ONNX TEST TARGETS PASS: True
ONNX POST-OPTIMIZATION END-TO-END TEST: SAVED AND FROZEN


In [8]:
for result in first_round_results[:6]:
    print('=' * 80)
    print('Event:', result['event_id'])
    print('Research label for QA only:', result['research_original_label'])
    print('Decision:', result['predicted_class'], '| Confidence:', f"{result['detector_confidence']:.6f}")
    print('Strict valid:', result['strict_valid'], '| End-to-End ms:', f"{result['end_to_end_ms']:.6f}")
    print(result['explanation'])
print('ONNX POST-OPTIMIZATION END-TO-END HUMAN QA EXAMPLES: REVIEWED')


Event: test-000175
Research label for QA only: FTP-BruteForce
Decision: Attack | Confidence: 0.999780
Strict valid: True | End-to-End ms: 6.762974
Summary: Attack with 1.000 confidence
Evidence: Fwd Seg Size Min and Init Fwd Win Byts provided the strongest local SHAP support for the detector decision.
Action: Review network, authentication, and host logs, corroborate before containment.
Event: test-000528
Research label for QA only: FTP-BruteForce
Decision: Attack | Confidence: 0.999780
Strict valid: True | End-to-End ms: 6.527903
Summary: Attack with 1.000 confidence
Evidence: Fwd Seg Size Min and Init Fwd Win Byts provided the strongest local SHAP support for the detector decision.
Action: Review network, authentication, and host logs, corroborate before containment.
Event: test-000733
Research label for QA only: FTP-BruteForce
Decision: Attack | Confidence: 0.999780
Strict valid: True | End-to-End ms: 6.692681
Summary: Attack with 1.000 confidence
Evidence: Fwd Seg Size Min and Init

## Post-optimization interpretation rule

- The primary claims are measured detector P95 and complete `end_to_end_ms` P95 across 1,220 single-event calls.
- File loading and model, TreeExplainer, and ONNX Runtime initialization are cold-start costs and are reported separately.
- A passing result requires detector P95 below 10 ms, complete-path P95 below 50 ms, numerical equivalence, 100% fidelity, and zero fallback.
- The NF4 model remains a separately disclosed fallback and its latency is not hidden.
- Steps 11D and 11F remain the earlier Test measurements. Step 11H must be labeled post-optimization and confirmatory.
